<a href="https://colab.research.google.com/github/RafaelCaballero/BME/blob/main/regreIbex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Ejemplo Regresión

### Carga de datos

In [8]:
import yfinance as yf
import pandas as pd


def descargar_cierres_ajustados(
    fecha_inicio="2021-01-01",
    fecha_fin=None
):
    """
    Descarga datos diarios de Yahoo Finance desde fecha_inicio
    y devuelve un DataFrame con los cierres ajustados de:
    Inditex, Iberdrola, Santander y BBVA.
    """

    tickers = {
        "Inditex": "ITX.MC",
        "Iberdrola": "IBE.MC",
        "Santander": "SAN.MC",
        "BBVA": "BBVA.MC"
    }

    datos = yf.download(
        list(tickers.values()),
        start=fecha_inicio,
        end=fecha_fin,
        interval="1d",
        auto_adjust=False,
        progress=False
    )

    cierres_ajustados = datos["Adj Close"]

    # Cambiar nombres de columnas de ticker a nombre de empresa
    cierres_ajustados = cierres_ajustados.rename(
        columns={v: k for k, v in tickers.items()}
    )

    return cierres_ajustados


df = descargar_cierres_ajustados()
df

Ticker,BBVA,Iberdrola,Inditex,Santander
Date,,,,
2021-01-04,2.955175,9.470932,21.430626,2.123436
2021-01-05,2.952277,9.323755,21.355982,2.130882
2021-01-06,3.122572,9.729481,21.895065,2.277326
2021-01-07,3.153733,9.773234,21.754074,2.310835
2021-01-08,3.098658,9.948256,22.160461,2.283531
...,...,...,...,...
2026-06-22,21.750000,21.190001,55.160000,11.998000
2026-06-23,21.690001,21.180000,55.160000,11.930000
2026-06-24,21.320000,21.150000,55.779999,11.862000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1402 entries, 2021-01-04 to 2026-06-26
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   BBVA       1402 non-null   float64
 1   Iberdrola  1402 non-null   float64
 2   Inditex    1402 non-null   float64
 3   Santander  1402 non-null   float64
dtypes: float64(4)
memory usage: 54.8 KB


### Preparación de datos

In [9]:
df_rend = df.pct_change().dropna()
df_rend

Ticker,BBVA,Iberdrola,Inditex,Santander
Date,,,,
2021-01-05,-0.000981,-0.015540,-0.003483,0.003507
2021-01-06,0.057683,0.043515,0.025243,0.068725
2021-01-07,0.009979,0.004497,-0.006439,0.014714
2021-01-08,-0.017463,0.017908,0.018681,-0.011815
2021-01-11,-0.005847,-0.007597,-0.018713,0.018116
...,...,...,...,...
2026-06-22,0.019213,0.017283,-0.010405,0.016607
2026-06-23,-0.002759,-0.000472,0.000000,-0.005668
2026-06-24,-0.017059,-0.001416,0.011240,-0.005700


### Obtención del error

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
import math
from tqdm import tqdm

def evalua_regresion(df,XColumns,yColumn,veces=500):
    # 1
    X = df[XColumns]
    y = df[yColumn]

    resultados = []
    todos_los_errores = []

    for v in tqdm(range(veces)):
        # 2
        test = 0.4
        X_train, X_cal, y_train, y_cal = train_test_split(X, y, test_size= test)

        # 3
        metodo = LinearRegression()
        modelo = metodo.fit(X_train,y_train)

        # 4
        y_pred = modelo.predict(X_cal)
        r2 = r2_score(y_cal,y_pred)
        rmse = math.sqrt(mean_squared_error(y_cal,y_pred))
        mae = mean_absolute_error(y_cal,y_pred)
        bias = (y_cal - y_pred).mean()
        resultados.append([round(r2,3),round(rmse,3),round(mae,3),round(bias,3)])

        # 5. Errores absolutos fuera de muestra para el intervalo predictivo
        errores_abs = np.abs(y_cal - y_pred)
        todos_los_errores.extend(errores_abs)

    df_errores = pd.DataFrame(resultados,columns=["r^2","RMSE","MAE","BIAS"])

    resumen = df_errores.describe().loc["mean"]
    resumen["q"] = np.quantile(todos_los_errores, 0.95)

    return resumen


In [11]:
df_rend.columns

Index(['BBVA', 'Iberdrola', 'Inditex', 'Santander'], dtype='object', name='Ticker')

In [15]:
X_columns = ['Iberdrola', 'Inditex', 'Santander']
y_column = "BBVA"

errores = evalua_regresion(df_rend,X_columns,y_column,veces=500)
errores


100%|██████████| 500/500 [00:04<00:00, 118.89it/s]


,mean
r^2,0.646268
RMSE,0.011576
MAE,0.008036
BIAS,-0.000044
q,0.020197
